# V2 static inference with restored hierarchical referencesSealed static-test inference for the V2 hierarchical geometry model: independent patch/file energies from frozen references, conformal anomaly confidence calibrated on validation data only, timestep localization, and family/severity slices with provenance-separated bounded outputs.Key features:- Direct parameters: configure inference via `InferenceParams` (e.g. `params = InferenceParams(max_files=512)`) or `V2_DATA_ROOT` / `V2_CHECKPOINT_PATH` / `V2_OUTPUT_ROOT` environment variables.- Manual paths: run from the repository root and set `SRC_DIR`, `V2_DATA_ROOT`, and `V2_CHECKPOINT_PATH` in the first cells; the exact configured paths are used with fail-fast errors naming the variable to change.- Restored references by default: `V2InferencePipeline.load` uses the checkpoint geometry as-is and fails fast when the checkpoint is absent. Refit happens only with `V2_REFIT_REFERENCES=true`, on verified-healthy development files, and is recorded in every output; the checkpoint bank file is never overwritten silently.- Independent energies: patch-level Mahalanobis mixture energy plus distribution-preserving file states (`tail_energy`, `elevated_fraction`, quantiles) are reported separately before any decision.- Validation-only calibration: the restored conformal calibrator (fit on validation displacements) maps file states to anomaly confidence; test labels only score the operating point, never fit it.- Localization and slices: top-tail patch-to-timestep mapping with anomaly-mask overlap on a bounded probe, plus family/severity breakdowns and false-positive behavior.- Provenance-separated bounded outputs: `metrics.json`, `slices.csv`, and `provenance.json` carry checkpoint, data-manifest, reference-source, and calibration-source records; bulk scores stay in memory.

In [ ]:
import csv
import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path

import torch

os.environ.setdefault("PYTHONHASHSEED", "0")

# ---- Repository source (edit these one-line values; used exactly) ----
# Run notebooks from the repository root, or set V2_REPO_ROOT to the checkout path.
V2_REPO_ROOT = os.environ.get("V2_REPO_ROOT", ".")
SRC_DIR = os.environ.get("V2_SRC_DIR", str(Path(V2_REPO_ROOT) / "src"))

_source_dir = Path(SRC_DIR).expanduser()
if not (_source_dir / "representation").is_dir():
    raise FileNotFoundError(
        f"Repository source not found: SRC_DIR={_source_dir} has no 'representation' package. "
        "Run from the repository root or set V2_REPO_ROOT / V2_SRC_DIR to the checkout.")
_source_resolved = str(_source_dir.resolve())
if _source_resolved not in sys.path:
    sys.path.insert(0, _source_resolved)
print(f"[Env] Loaded representation modules from: {_source_dir}")

from representation.data import collate_variable_files
from representation.v2_checkpoint import load_v2_checkpoint
from representation.v2_inference import V2InferencePipeline, patch_regime_ids
from representation.v2_trajectory import TrajectoryTracker
from synth.chronicle import load_chronological
from synth.config import PatchConfig
from synth.patchify import Patchifier
from synth.schema import SampleLabel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[Hardware] Compute device:", device)

In [ ]:
# ---- Explicit data / checkpoint / output roots (edit these one-line values; used exactly) ----
V2_DATA_ROOT = os.environ.get("V2_DATA_ROOT", "data/generated/chronicle-client")
V2_CHECKPOINT_PATH = os.environ.get("V2_CHECKPOINT_PATH", "checkpoints/v2_geometry.pt")
V2_OUTPUT_ROOT = os.environ.get("V2_OUTPUT_ROOT", "outputs/v2_static")
# Opt-in only: replace restored references with a dev-train refit (recorded; never silent).
REFIT_REFERENCES = os.environ.get("V2_REFIT_REFERENCES", "false").lower() in ("true", "1", "yes")

@dataclass
class InferenceParams:
    """V2 static inference configuration (works from a repository checkout)."""
    data_root: str = V2_DATA_ROOT
    checkpoint_path: str = V2_CHECKPOINT_PATH
    output_root: str = V2_OUTPUT_ROOT
    batch_size: int = int(os.environ.get("V2_BATCH_SIZE", "8"))
    max_files: int | None = int(os.environ["V2_MAX_FILES"]) if "V2_MAX_FILES" in os.environ else None
    refit_references: bool = REFIT_REFERENCES
    seed: int = int(os.environ.get("V2_SEED", "0"))

params = InferenceParams()
configured_root = Path(params.data_root).expanduser()
DATA_ROOT = configured_root if configured_root.is_absolute() else Path.cwd() / configured_root
MANIFEST_PATH = DATA_ROOT / "manifest.json"
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Chronological manifest not found at {MANIFEST_PATH}. Set V2_DATA_ROOT to the materialized "
        "chronicle root (generate it with notebooks/generate_chronological_factory.ipynb).")
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
configured_ckpt = Path(params.checkpoint_path).expanduser()
CHECKPOINT_PATH = configured_ckpt if configured_ckpt.is_absolute() else Path.cwd() / configured_ckpt
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"V2 checkpoint not found at {CHECKPOINT_PATH}. Set V2_CHECKPOINT_PATH to the trained "
        "V2 checkpoint (see notebooks/train_v2_geometry.ipynb). Inference never scores without it.")
print(f"[Config] data={DATA_ROOT} checkpoint={CHECKPOINT_PATH} batch_size={params.batch_size} "
      f"max_files={params.max_files} refit={params.refit_references}")

In [ ]:
# Restore the pipeline with hierarchical references exactly as saved.
pipeline = V2InferencePipeline.load(CHECKPOINT_PATH, device=str(device))
payload = load_v2_checkpoint(CHECKPOINT_PATH, pipeline.model, expected_config=pipeline.config)
if not isinstance(payload.get("geometry"), dict):
    raise RuntimeError(f"Checkpoint at {CHECKPOINT_PATH} carries no fitted geometry references.")
tracker_state = payload.get("tracker")
if not isinstance(tracker_state, dict):
    raise RuntimeError(f"Checkpoint at {CHECKPOINT_PATH} carries no commissioned trajectory baseline.")
print(f"[Provenance] fallback_order={pipeline.config.fallback_order()} "
      f"calibrator={'restored' if pipeline.calibrator is not None else 'absent'} "
      f"risk={'restored' if pipeline.risk is not None else 'absent'} "
      f"elevated_threshold={pipeline.elevated_threshold}")

samples, manifest = load_chronological(DATA_ROOT)
by_id = {s.file_id: s for s in samples}
if "test_static" not in manifest["splits"] or not manifest["splits"]["test_static"]:
    raise RuntimeError(f"Chronicle static test at {DATA_ROOT} is empty; regenerate the dataset.")
static_ids = manifest["splits"]["test_static"]
if params.max_files is not None:
    static_ids = static_ids[:params.max_files]
static_files = [by_id[i] for i in static_ids]
patchifier = Patchifier(PatchConfig(patch_size=pipeline.config.patch_size, stride=pipeline.config.stride))

reference_source = "restored-checkpoint"
if params.refit_references:
    dev_ids = manifest["splits"]["dev_train"]
    if not dev_ids:
        raise RuntimeError("Explicit refit requested but the dev-train view is empty.")
    dev_files = [by_id[i] for i in dev_ids
                 if by_id[i].file_label is SampleLabel.NORMAL
                 and (by_id[i].split_provenance is None or not by_id[i].split_provenance.is_quarantined)]
    if not dev_files:
        raise RuntimeError("Explicit refit requested but no verified-healthy dev-train files exist.")
    pipeline.model.eval()
    ref_latents, ref_robot, ref_program, ref_regime = [], [], [], []
    with torch.no_grad():
        for start in range(0, len(dev_files), params.batch_size):
            chunk = dev_files[start:start + params.batch_size]
            base = collate_variable_files(chunk, patchifier)
            count = base["patches"].shape[1]
            regimes = patch_regime_ids(chunk, base["starts"], count)
            encoded = pipeline.model(base["patches"].to(device), base["patch_pad_mask"].to(device),
                                     base["patch_valid_mask"].to(device), base["robot_idx"].to(device),
                                     base["program_idx"].to(device), regimes.to(device))
            valid = base["patch_valid_mask"]
            ref_latents.append(encoded["patch_latents"].cpu()[valid])
            ref_robot.append(base["robot_idx"].unsqueeze(1).expand_as(valid)[valid])
            ref_program.append(base["program_idx"].unsqueeze(1).expand_as(valid)[valid])
            ref_regime.append(regimes[valid])
    pipeline.refit_references(torch.cat(ref_latents), torch.cat(ref_robot),
                              torch.cat(ref_program), torch.cat(ref_regime),
                              torch.ones(sum(r.shape[0] for r in ref_latents), dtype=torch.bool))
    reference_source = f"explicit-refit-on-{len(dev_files)}-dev-train-files"
print(f"[References] source={reference_source}; the checkpoint file itself was not modified.")

In [ ]:
# Score the sealed static test: independent patch/file energies, confidence, localization.
scoring_tracker = TrajectoryTracker(pipeline.config.d_model)
scoring_tracker.load_state_dict(tracker_state)
file_rows, patch_energy_list, confidence_list = [], [], []
family_names, severity_names, file_labels, file_ids = [], [], [], []
for start in range(0, len(static_files), params.batch_size):
    chunk = static_files[start:start + params.batch_size]
    base = collate_variable_files(chunk, patchifier)
    count = base["patches"].shape[1]
    regimes = patch_regime_ids(chunk, base["starts"], count)
    out = pipeline.score_patches(base["patches"], base["patch_pad_mask"], base["patch_valid_mask"],
                                 base["robot_idx"], base["program_idx"], regimes,
                                 tracker=scoring_tracker)
    file_rows.append(out["file"]["file_state"])
    patch_energy_list.append(out["patch"]["patch_energy"])
    if out["confidence"]:
        confidence_list.append(out["confidence"]["confidence"])
    for sample in chunk:
        file_ids.append(sample.file_id)
        file_labels.append(sample.file_label is SampleLabel.ABNORMAL)
        meta = sample.anomaly_meta
        family_names.append(meta.family.value if meta is not None else "normal")
        severity_names.append(str(getattr(meta, "severity", "unknown")) if meta is not None else "healthy")
file_states = torch.cat(file_rows)
# File-state fields (tail_energy, elevated_fraction, quantiles) are read from the
# validated file-state contract inside score_patches; no second aggregation lives here.
# Independent file signals come straight from the validated file-state contract.
print(f"[Scores] files={file_states.shape[0]} patch_batches={len(patch_energy_list)} "
      f"confidence={'restored-validator' if confidence_list else 'absent (checkpoint has no calibrator)'}")
if not confidence_list:
    raise RuntimeError("Static inference requires the validation-calibrated confidence layer; "
                       "retrain with notebooks/train_v2_geometry.ipynb calibration.")
confidences = torch.cat(confidence_list)
patch_energies = patch_energy_list
# Localization probe: top-tail patches mapped to timesteps on the first 3 abnormal files.
localized = 0
for sample in [s for s in static_files if s.file_label is SampleLabel.ABNORMAL][:3]:
    base = collate_variable_files([sample], patchifier)
    count = base["patches"].shape[1]
    regimes = patch_regime_ids([sample], base["starts"], count)
    probe_tracker = TrajectoryTracker(pipeline.config.d_model)
    probe_tracker.load_state_dict(tracker_state)
    out = pipeline.score_patches(base["patches"], base["patch_pad_mask"], base["patch_valid_mask"],
                                 base["robot_idx"], base["program_idx"], regimes, tracker=probe_tracker)
    energies = out["patch"]["patch_energy"][0]
    valid = out["patch"]["patch_valid_mask"][0]
    order = torch.argsort(energies[valid], descending=True)
    top = order[:3].tolist()
    valid_idx = torch.nonzero(valid).reshape(-1).tolist()
    starts = base["starts"][0].tolist()
    width, stride = pipeline.config.patch_size, pipeline.config.stride
    ranges = [(starts[valid_idx[k]], starts[valid_idx[k]] + width) for k in top]
    overlap = "no-mask"
    if sample.anomaly_mask is not None:
        flagged = sample.anomaly_mask.any(axis=0)
        covered = sum(flagged[max(0, a):b].sum() for a, b in ranges)
        span = sum(b - max(0, a) for a, b in ranges)
        overlap = f"{float(covered) / max(1, span):.2f}"
    print(f"[Localization] {sample.file_id} top_patches={top} timestep_ranges={ranges} mask_overlap={overlap}")
    localized += 1
print(f"[Localization] probe covered {localized} abnormal files; full patch energies stay in memory.")

In [ ]:
# Validation-calibrated operating point, family/severity slices, and bounded outputs.
from sklearn.metrics import average_precision_score, roc_auc_score

if "dev_val" not in manifest["splits"] or not manifest["splits"]["dev_val"]:
    raise RuntimeError("Validation-only calibration needs a non-empty dev-val view.")
val_probe_files = [by_id[i] for i in manifest["splits"]["dev_val"][:64]]
val_tracker = TrajectoryTracker(pipeline.config.d_model)
val_tracker.load_state_dict(tracker_state)
val_confidences = []
for start in range(0, len(val_probe_files), params.batch_size):
    chunk = val_probe_files[start:start + params.batch_size]
    base = collate_variable_files(chunk, patchifier)
    count = base["patches"].shape[1]
    regimes = patch_regime_ids(chunk, base["starts"], count)
    out = pipeline.score_patches(base["patches"], base["patch_pad_mask"], base["patch_valid_mask"],
                                 base["robot_idx"], base["program_idx"], regimes, tracker=val_tracker)
    if not out["confidence"]:
        raise RuntimeError("Restored calibrator produced no confidence on validation data.")
    val_confidences.append(out["confidence"]["confidence"])
val_confidences = torch.cat(val_confidences)
operating_threshold = float(torch.quantile(val_confidences, 0.95))
print(f"[Calibration] validation-only operating threshold={operating_threshold:.4f} "
      f"(95th percentile over {val_confidences.numel()} dev-val confidences; test labels never fit it).")

truth = torch.tensor(file_labels, dtype=bool)
decisions = confidences >= operating_threshold
tp = int((decisions & truth).sum()); fp = int((decisions & ~truth).sum())
fn = int(((~decisions) & truth).sum()); tn = int(((~decisions) & ~truth).sum())
precision = tp / max(1, tp + fp)
recall = tp / max(1, tp + fn)
f1 = 2 * precision * recall / max(1e-6, precision + recall)
auroc = float(roc_auc_score(truth.tolist(), confidences.tolist())) if tp + fn > 0 and tn + fp > 0 else float("nan")
auprc = float(average_precision_score(truth.tolist(), confidences.tolist())) if tp + fn > 0 else float("nan")
print(f"[Metrics] n={len(truth)} abnormal={int(truth.sum())} AUROC={auroc:.4f} AUPRC={auprc:.4f} "
      f"F1={f1:.4f} (P={precision:.4f} R={recall:.4f}) FP={fp} TN={tn}")
from collections import defaultdict
family_hits: dict[str, list[bool]] = defaultdict(list)
severity_hits: dict[str, list[bool]] = defaultdict(list)
for family, severity, decision, label in zip(family_names, severity_names, decisions.tolist(), file_labels):
    if label:
        family_hits[family].append(decision)
    severity_hits[f"{family}/{severity}" if label else "normal/healthy"].append(decision)
for family in sorted(family_hits):
    hits = family_hits[family]
    print(f"[Slice] family {family}: recalled {sum(hits)}/{len(hits)}")
OUTPUT_DIR = Path(params.output_root).expanduser()
OUTPUT_DIR = OUTPUT_DIR if OUTPUT_DIR.is_absolute() else Path.cwd() / OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics = {"n_files": len(truth), "n_abnormal": int(truth.sum()), "auroc": auroc, "auprc": auprc,
           "f1": f1, "precision": precision, "recall": recall, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
           "operating_threshold": operating_threshold,
           "calibration_source": "dev-val confidences only (restored calibrator)",
           "reference_source": reference_source}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, sort_keys=True), encoding="utf-8")
with (OUTPUT_DIR / "slices.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["slice", "kind", "n", "flagged"])
    for family in sorted(family_hits):
        hits = family_hits[family]
        writer.writerow([family, "family", len(hits), sum(hits)])
    for key in sorted(severity_hits):
        hits = severity_hits[key]
        writer.writerow([key, "severity", len(hits), sum(hits)])
provenance = {"data_root": str(DATA_ROOT), "manifest_path": str(MANIFEST_PATH),
              "config_hash": manifest["config_hash"], "checkpoint": str(CHECKPOINT_PATH),
              "checkpoint_config": pipeline.config.to_dict(), "reference_source": reference_source,
              "calibration_source": "dev-val confidences only (restored calibrator)",
              "static_file_ids": file_ids}
(OUTPUT_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2, sort_keys=True), encoding="utf-8")
print(f"[Output] wrote metrics.json, slices.csv, provenance.json to {OUTPUT_DIR} "
      "(reference_source and calibration_source recorded; bulk scores not persisted).")